# Notebook 08 — Sodium Case-Study Candidate Triage

**Project:** CMT Path A — leakage-audited multi-ion computed insertion-electrode benchmark

**Purpose:** build a cautious sodium-ion case-study triage table using outputs from Notebooks 08–11.

This notebook performs sodium candidate triage using computed MP electrode records, Pareto membership, rank robustness, uncertainty penalties, and applicability-domain penalties.

**Strict exclusions:**
- no ML training,
- no CDE matching,
- no criticality filtering,
- no manuscript writing,
- no DFT candidate selection.

The output is a **case-study shortlist for manual literature-analogue review in Notebook 09**, not a discovery claim.

In [ ]:
# ============================================================
# Cell 1 — Imports, paths, logging, and notebook identity
# ============================================================

from __future__ import annotations

import os
import sys
import re
import json
import math
import platform
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.options.mode.copy_on_write = True

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# -
# Locate prior notebook outputs
# -
def find_output_dir(name: str) -> Path:
    candidates = [
        Path("outputs") / name,
        Path(name),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find {name}. Expected one of:\n"
        f"  outputs/{name}/\n"
        f"  {name}/"
    )

NB08_DIR = find_output_dir("Notebook 01")
NB09_DIR = find_output_dir("Notebook 02")
NB10_DIR = find_output_dir("Notebook 04")
NB11_DIR = find_output_dir("Notebook 07")

# -
# Notebook 08 output folders
# -
BASE_DIR = Path("outputs") / "Notebook 08"
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"

for d in [BASE_DIR, PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "08_event_log.csv", index=False)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

def safe_float(x):
    try:
        if x is None or x == "":
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def safe_str(x):
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except Exception:
        pass
    return str(x)

log_event("init", "INFO", "Notebook 08 initialized.", {
    "NB08_DIR": str(NB08_DIR),
    "NB09_DIR": str(NB09_DIR),
    "NB10_DIR": str(NB10_DIR),
    "NB11_DIR": str(NB11_DIR),
    "BASE_DIR": str(BASE_DIR),
})

print("Using prior outputs:")
print(f"  Notebook 01: {NB08_DIR}")
print(f"  Notebook 02: {NB09_DIR}")
print(f"  Notebook 04: {NB10_DIR}")
print(f"  Notebook 07: {NB11_DIR}")
print(f"Notebook 08 outputs: {BASE_DIR}")


In [ ]:
# ============================================================
# Cell 2 — Notebook 08 configuration
# ============================================================

# Main sodium case-study protocol.
# P3 is the post-DFT decision-support clean protocol from Notebook 02.
CASE_STUDY_PROTOCOL = "P3"

# Main split for cross-ion uncertainty transfer diagnostics.
CASE_STUDY_SPLIT = "leave_working_ion_out"

# Core targets for the sodium case study.
TRIAGE_TARGETS = [
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_worst",
]

# Conservative case-study filters, reused only as transparent triage thresholds.
# These are NOT criticality filters and NOT discovery claims.
CASE_STUDY_THRESHOLDS = {
    "average_voltage_min": 2.0,
    "average_voltage_max": 4.5,
    "capacity_grav_min": 50.0,
    "energy_grav_min": 150.0,
    "max_delta_volume_max": 0.35,
    "stability_worst_max": 0.10,
}

# Score normalization ranges. These define a transparent decision-support score.
# They are not physical laws and must be reported as triage settings.
SCORE_RANGES = {
    "average_voltage": {"low": 2.0, "high": 4.5, "direction": "benefit"},
    "capacity_grav": {"low": 50.0, "high": 250.0, "direction": "benefit"},
    "energy_grav": {"low": 150.0, "high": 900.0, "direction": "benefit"},
    "max_delta_volume": {"good": 0.0, "bad": 0.35, "direction": "cost"},
    "stability_worst": {"good": 0.0, "bad": 0.10, "direction": "cost"},
}

SCORE_WEIGHTS = {
    "average_voltage": 0.20,
    "capacity_grav": 0.25,
    "energy_grav": 0.30,
    "max_delta_volume": 0.10,
    "stability_worst": 0.15,
}

# Monte-Carlo rank robustness using conformal interval widths.
RANK_MC_SAMPLES = 500
RANDOM_SEED = 42

# Do not make candidate discovery claims.
NOTEBOOK_08_ASSERTIONS = {
    "ml_training_performed": False,
    "ranking_performed": True,
    "ranking_type": "transparent sodium case-study triage only",
    "cde_matching_performed": False,
    "criticality_filtering_performed": False,
    "manuscript_writing_performed": False,
    "dft_candidate_selection_performed": False,
    "discovery_claim_made": False,
}

write_json_safe(
    {
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "case_study_protocol": CASE_STUDY_PROTOCOL,
        "case_study_split": CASE_STUDY_SPLIT,
        "triage_targets": TRIAGE_TARGETS,
        "case_study_thresholds": CASE_STUDY_THRESHOLDS,
        "score_ranges": SCORE_RANGES,
        "score_weights": SCORE_WEIGHTS,
        "rank_mc_samples": RANK_MC_SAMPLES,
        "random_seed": RANDOM_SEED,
        "assertions": NOTEBOOK_08_ASSERTIONS,
    },
    METADATA_DIR / "08_config.json",
)

write_json_safe(NOTEBOOK_08_ASSERTIONS, AUDIT_DIR / "08_no_cde_criticality_assertion.json")

display(pd.DataFrame([NOTEBOOK_08_ASSERTIONS]))


In [ ]:
# ============================================================
# Cell 3 — Load previous final decisions and required inputs
# ============================================================

def load_json_if_exists(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

previous_decisions = {
    "Notebook 01": load_json_if_exists(NB08_DIR / "metadata" / "01_final_decision.json"),
    "Notebook 02": load_json_if_exists(NB09_DIR / "metadata" / "02_final_decision.json"),
    "Notebook 04": load_json_if_exists(NB10_DIR / "metadata" / "04_final_decision.json"),
    "Notebook 07": load_json_if_exists(NB11_DIR / "metadata" / "07_final_decision.json"),
}

write_json_safe(previous_decisions, METADATA_DIR / "08_previous_notebook_decisions.json")

# Required paths
metadata_targets_path = NB09_DIR / "processed" / "02_master_metadata_and_targets.csv"
mask_path = NB09_DIR / "audit" / "02_target_plausibility_masks.csv"
feature_table_path = NB09_DIR / "processed" / "02_master_feature_table.csv"
uncertainty_predictions_path = NB11_DIR / "processed" / "07_uncertainty_predictions.csv"
uncertainty_summary_path = NB11_DIR / "processed" / "07_compact_uncertainty_ad_table.csv"
benchmark_compact_path = NB10_DIR / "processed" / "04_compact_best_model_benchmark_table.csv"

required_paths = [
    metadata_targets_path,
    mask_path,
    feature_table_path,
    uncertainty_predictions_path,
    uncertainty_summary_path,
    benchmark_compact_path,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing_paths))

metadata_targets_df = pd.read_csv(metadata_targets_path, low_memory=False)
mask_df = pd.read_csv(mask_path, low_memory=False)

# Feature table is used only for extra non-leaky metadata and formula/feature support.
feature_df = pd.read_csv(feature_table_path, low_memory=False)

# Load compact benchmark summaries for context, not for ranking.
benchmark_compact_df = pd.read_csv(benchmark_compact_path, low_memory=False)
uncertainty_summary_df = pd.read_csv(uncertainty_summary_path, low_memory=False)

print("Loaded inputs:")
print(f"metadata_targets_df: {metadata_targets_df.shape}")
print(f"mask_df:             {mask_df.shape}")
print(f"feature_df:          {feature_df.shape}")
print(f"benchmark_compact:   {benchmark_compact_df.shape}")
print(f"uncertainty_summary: {uncertainty_summary_df.shape}")

decision_rows = []
for nb, d in previous_decisions.items():
    decision_rows.append({
        "notebook": nb,
        "final_decision": d.get("final_decision", "missing_or_unknown"),
    })
previous_decision_df = pd.DataFrame(decision_rows)
previous_decision_df.to_csv(AUDIT_DIR / "08_previous_decision_audit.csv", index=False)
display(previous_decision_df)


In [ ]:
# ============================================================
# Cell 4 — Build sodium base table and merge masks
# ============================================================

# Merge masks into metadata/targets.
base_df = metadata_targets_df.merge(
    mask_df.drop(columns=[c for c in mask_df.columns if c in metadata_targets_df.columns and c != "record_index"], errors="ignore"),
    on="record_index",
    how="left",
)

# Add selected columns from master feature table if present.
extra_feature_cols = [
    "record_index",
    "fracA_charge",
    "fracA_discharge",
    "num_steps",
    "max_voltage_step",
    "summary_worst_energy_above_hull",
    "charge_summary_energy_above_hull",
    "discharge_summary_energy_above_hull",
    "charge_summary_formula_pretty",
    "discharge_summary_formula_pretty",
]
extra_feature_cols = [c for c in extra_feature_cols if c in feature_df.columns]
if len(extra_feature_cols) > 1:
    base_df = base_df.merge(feature_df[extra_feature_cols].drop_duplicates("record_index"), on="record_index", how="left")

sodium_df = base_df[base_df["working_ion"].astype(str) == "Na"].copy()

# Numeric conversions
for col in TRIAGE_TARGETS + ["capacity_vol", "energy_vol", "stability_charge", "stability_discharge", "fracA_charge", "fracA_discharge", "num_steps", "max_voltage_step"]:
    if col in sodium_df.columns:
        sodium_df[col] = pd.to_numeric(sodium_df[col], errors="coerce")

# Case-study filters.
sodium_df["case_voltage_window_pass"] = sodium_df["average_voltage"].between(
    CASE_STUDY_THRESHOLDS["average_voltage_min"],
    CASE_STUDY_THRESHOLDS["average_voltage_max"],
    inclusive="both",
)
sodium_df["case_capacity_pass"] = sodium_df["capacity_grav"] >= CASE_STUDY_THRESHOLDS["capacity_grav_min"]
sodium_df["case_energy_pass"] = sodium_df["energy_grav"] >= CASE_STUDY_THRESHOLDS["energy_grav_min"]
sodium_df["case_volume_pass"] = sodium_df["max_delta_volume"] <= CASE_STUDY_THRESHOLDS["max_delta_volume_max"]
sodium_df["case_stability_pass"] = sodium_df["stability_worst"] <= CASE_STUDY_THRESHOLDS["stability_worst_max"]

# Prefer Notebook 02 all-target physical plausibility where available.
if "mask_physics_plausible_all_targets" in sodium_df.columns:
    sodium_df["case_physics_plausible"] = sodium_df["mask_physics_plausible_all_targets"].fillna(False).astype(bool)
elif "mask_physics_plausible_primary" in sodium_df.columns:
    sodium_df["case_physics_plausible"] = sodium_df["mask_physics_plausible_primary"].fillna(False).astype(bool)
else:
    sodium_df["case_physics_plausible"] = True

sodium_df["case_screen_pass"] = sodium_df[
    [
        "case_physics_plausible",
        "case_voltage_window_pass",
        "case_capacity_pass",
        "case_energy_pass",
        "case_volume_pass",
        "case_stability_pass",
    ]
].all(axis=1)

# Save base sodium table before uncertainty ranking.
base_path = PROCESSED_DIR / "08_sodium_candidate_base_table.csv"
sodium_df.to_csv(base_path, index=False)

# Screening funnel.
funnel_rows = [
    {"stage": "All Na insertion-electrode records", "n_records": len(sodium_df)},
    {"stage": "Physics-plausible all/primary mask", "n_records": int(sodium_df["case_physics_plausible"].sum())},
    {"stage": "Voltage 2.0–4.5 V", "n_records": int(sodium_df["case_voltage_window_pass"].sum())},
    {"stage": "Capacity ≥ 50 mAh/g", "n_records": int(sodium_df["case_capacity_pass"].sum())},
    {"stage": "Energy ≥ 150 Wh/kg", "n_records": int(sodium_df["case_energy_pass"].sum())},
    {"stage": "Volume change ≤ 0.35", "n_records": int(sodium_df["case_volume_pass"].sum())},
    {"stage": "Worst stability ≤ 0.10 eV/atom", "n_records": int(sodium_df["case_stability_pass"].sum())},
    {"stage": "All case-study screen criteria", "n_records": int(sodium_df["case_screen_pass"].sum())},
]
funnel_df = pd.DataFrame(funnel_rows)
funnel_df["pct_of_Na_total"] = 100.0 * funnel_df["n_records"] / max(len(sodium_df), 1)
funnel_df.to_csv(AUDIT_DIR / "08_sodium_screening_funnel.csv", index=False)

input_coverage_df = pd.DataFrame([{
    "n_sodium_records": len(sodium_df),
    "n_case_screen_pass": int(sodium_df["case_screen_pass"].sum()),
    "n_unique_framework_uid": sodium_df["framework_uid"].nunique(dropna=True),
    "n_unique_chemical_system_uid": sodium_df["chemical_system_uid"].nunique(dropna=True),
    "n_unique_coarse_family": sodium_df["coarse_family"].nunique(dropna=True),
    "base_table": str(base_path),
}])
input_coverage_df.to_csv(AUDIT_DIR / "08_input_coverage_audit.csv", index=False)

display(input_coverage_df)
display(funnel_df)
print(f"Saved sodium base table: {base_path}")


In [ ]:
# ============================================================
# Cell 5 — Load and aggregate Notebook 07 uncertainty predictions for sodium
# ============================================================

USECOLS_11 = [
    "target",
    "protocol",
    "split_name",
    "fold_id",
    "heldout_group",
    "model_name",
    "record_index",
    "electrode_uid",
    "working_ion",
    "true_value",
    "predicted_value",
    "absolute_error",
    "tree_pred_std",
    "conformal_lower",
    "conformal_upper",
    "conformal_width",
    "conformal_covered",
    "knn_mean_distance",
    "knn_min_distance",
    "domain_distance_percentile",
    "ad_flag",
    "n_train",
    "n_test",
    "n_features",
]

# Read only columns available in the file.
header_cols = pd.read_csv(uncertainty_predictions_path, nrows=0).columns.tolist()
usecols = [c for c in USECOLS_11 if c in header_cols]
uncertainty_preds_df = pd.read_csv(uncertainty_predictions_path, usecols=usecols, low_memory=False)

# Restrict to Na, leave-working-ion-out, primary targets, and protocol P1/P2/P3.
uncertainty_na_df = uncertainty_preds_df[
    (uncertainty_preds_df["working_ion"].astype(str) == "Na")
    & (uncertainty_preds_df["split_name"].astype(str) == CASE_STUDY_SPLIT)
    & (uncertainty_preds_df["target"].isin(TRIAGE_TARGETS))
    & (uncertainty_preds_df["protocol"].isin(["P1", "P2", "P3"]))
].copy()

# If P3 is unavailable for any target, downstream code falls back target-by-target.
for col in ["true_value", "predicted_value", "absolute_error", "tree_pred_std", "conformal_lower", "conformal_upper", "conformal_width", "domain_distance_percentile"]:
    if col in uncertainty_na_df.columns:
        uncertainty_na_df[col] = pd.to_numeric(uncertainty_na_df[col], errors="coerce")

# AD flag severity aggregation.
ad_flag_to_num = {"in_domain": 0, "borderline": 1, "out_of_domain": 2}
ad_num_to_flag = {0: "in_domain", 1: "borderline", 2: "out_of_domain"}
uncertainty_na_df["ad_flag_num"] = uncertainty_na_df.get("ad_flag", "").map(ad_flag_to_num).fillna(1).astype(int)

prediction_coverage_rows = []
for target in TRIAGE_TARGETS:
    for protocol in ["P1", "P2", "P3"]:
        sub = uncertainty_na_df[(uncertainty_na_df["target"] == target) & (uncertainty_na_df["protocol"] == protocol)]
        prediction_coverage_rows.append({
            "target": target,
            "protocol": protocol,
            "n_prediction_rows": len(sub),
            "n_unique_electrodes": sub["electrode_uid"].nunique() if not sub.empty else 0,
            "pct_of_sodium_records": 100.0 * (sub["electrode_uid"].nunique() if not sub.empty else 0) / max(len(sodium_df), 1),
        })

prediction_coverage_df = pd.DataFrame(prediction_coverage_rows)
prediction_coverage_df.to_csv(AUDIT_DIR / "08_prediction_coverage_by_target.csv", index=False)

display(prediction_coverage_df)
print(f"Uncertainty rows loaded for Na case study: {len(uncertainty_na_df)}")


In [ ]:
# ============================================================
# Cell 6 — Target-wise protocol fallback and wide prediction table
# ============================================================

def choose_protocol_for_target(target: str) -> str:
    """Prefer CASE_STUDY_PROTOCOL; fallback to P2, then P1 if coverage is missing."""
    for protocol in [CASE_STUDY_PROTOCOL, "P2", "P1"]:
        sub = uncertainty_na_df[(uncertainty_na_df["target"] == target) & (uncertainty_na_df["protocol"] == protocol)]
        if sub["electrode_uid"].nunique() >= max(10, 0.5 * len(sodium_df)):
            return protocol
    return CASE_STUDY_PROTOCOL

selected_protocol_by_target = {target: choose_protocol_for_target(target) for target in TRIAGE_TARGETS}
write_json_safe(selected_protocol_by_target, METADATA_DIR / "08_selected_protocol_by_target.json")

selected_uncertainty_rows = []
for target, protocol in selected_protocol_by_target.items():
    sub = uncertainty_na_df[(uncertainty_na_df["target"] == target) & (uncertainty_na_df["protocol"] == protocol)].copy()
    selected_uncertainty_rows.append(sub)

if selected_uncertainty_rows:
    selected_uncertainty_df = pd.concat(selected_uncertainty_rows, ignore_index=True)
else:
    selected_uncertainty_df = pd.DataFrame()

# Aggregate if a record appears more than once.
agg_map = {
    "predicted_value": "mean",
    "true_value": "mean",
    "absolute_error": "mean",
    "tree_pred_std": "mean",
    "conformal_lower": "median",
    "conformal_upper": "median",
    "conformal_width": "median",
    "knn_mean_distance": "mean",
    "knn_min_distance": "min",
    "domain_distance_percentile": "mean",
    "ad_flag_num": "max",
}
agg_map = {k: v for k, v in agg_map.items() if k in selected_uncertainty_df.columns}

if selected_uncertainty_df.empty:
    pred_agg_df = pd.DataFrame(columns=["electrode_uid", "target"])
else:
    pred_agg_df = (
        selected_uncertainty_df
        .groupby(["electrode_uid", "record_index", "target"], dropna=False)
        .agg(agg_map)
        .reset_index()
    )
    pred_agg_df["selected_protocol"] = pred_agg_df["target"].map(selected_protocol_by_target)
    pred_agg_df["ad_flag_agg"] = pred_agg_df["ad_flag_num"].map(ad_num_to_flag)

# Build wide table.
wide_base = sodium_df[["electrode_uid", "record_index"]].drop_duplicates().copy()
wide_pred_df = wide_base.copy()

for target in TRIAGE_TARGETS:
    sub = pred_agg_df[pred_agg_df["target"] == target].copy() if not pred_agg_df.empty else pd.DataFrame()
    if sub.empty:
        continue
    rename = {
        "predicted_value": f"pred_{target}",
        "true_value": f"unc_true_{target}",
        "absolute_error": f"unc_abs_error_{target}",
        "tree_pred_std": f"unc_tree_std_{target}",
        "conformal_lower": f"conf_lower_{target}",
        "conformal_upper": f"conf_upper_{target}",
        "conformal_width": f"conf_width_{target}",
        "knn_mean_distance": f"knn_mean_distance_{target}",
        "domain_distance_percentile": f"domain_pct_{target}",
        "ad_flag_num": f"ad_flag_num_{target}",
        "ad_flag_agg": f"ad_flag_{target}",
        "selected_protocol": f"selected_protocol_{target}",
    }
    keep_cols = ["electrode_uid"] + [c for c in rename if c in sub.columns]
    sub2 = sub[keep_cols].rename(columns=rename)
    wide_pred_df = wide_pred_df.merge(sub2, on="electrode_uid", how="left")

prediction_wide_path = PROCESSED_DIR / "08_sodium_lwoo_prediction_uncertainty_wide.csv"
wide_pred_df.to_csv(prediction_wide_path, index=False)

# Merge into sodium table.
sodium_case_df = sodium_df.merge(
    wide_pred_df.drop(columns=["record_index"], errors="ignore"),
    on="electrode_uid",
    how="left",
)

# Prediction coverage flags.
pred_cols = [f"pred_{t}" for t in TRIAGE_TARGETS if f"pred_{t}" in sodium_case_df.columns]
sodium_case_df["prediction_coverage_n_targets"] = sodium_case_df[pred_cols].notna().sum(axis=1) if pred_cols else 0
sodium_case_df["prediction_coverage_complete_primary"] = sodium_case_df["prediction_coverage_n_targets"] >= len(TRIAGE_TARGETS)

selected_protocol_df = pd.DataFrame([
    {"target": k, "selected_protocol": v} for k, v in selected_protocol_by_target.items()
])
selected_protocol_df.to_csv(AUDIT_DIR / "08_selected_protocol_by_target.csv", index=False)

display(selected_protocol_df)
print(f"Saved wide prediction table: {prediction_wide_path}")


In [ ]:
# ============================================================
# Cell 7 — Scoring utilities and computed/predicted triage scores
# ============================================================

def norm_benefit(x, low, high):
    x = pd.to_numeric(x, errors="coerce")
    return ((x - low) / (high - low)).clip(lower=0, upper=1)

def norm_cost(x, good, bad):
    x = pd.to_numeric(x, errors="coerce")
    return (1.0 - ((x - good) / (bad - good))).clip(lower=0, upper=1)

def score_component(series, target: str):
    cfg = SCORE_RANGES[target]
    if cfg["direction"] == "benefit":
        return norm_benefit(series, cfg["low"], cfg["high"])
    return norm_cost(series, cfg["good"], cfg["bad"])

def weighted_score_from_columns(df: pd.DataFrame, prefix: str = "") -> pd.Series:
    parts = []
    weights = []
    for target, w in SCORE_WEIGHTS.items():
        col = f"{prefix}{target}" if prefix else target
        if col not in df.columns:
            continue
        comp = score_component(df[col], target)
        parts.append(comp.fillna(0.0) * w)
        weights.append(w)
    if not parts:
        return pd.Series(np.nan, index=df.index)
    denom = sum(weights)
    return sum(parts) / denom if denom > 0 else pd.Series(np.nan, index=df.index)

# Computed MP target score.
sodium_case_df["computed_target_score"] = weighted_score_from_columns(sodium_case_df, prefix="")

# Predicted transfer score from Notebook 07 leave-working-ion-out predictions.
pred_score_input = pd.DataFrame(index=sodium_case_df.index)
for target in TRIAGE_TARGETS:
    pred_col = f"pred_{target}"
    pred_score_input[f"pred_{target}"] = sodium_case_df[pred_col] if pred_col in sodium_case_df.columns else np.nan
sodium_case_df["predicted_transfer_score"] = weighted_score_from_columns(pred_score_input, prefix="pred_")

# If predictions are missing, keep a fallback only for audit. The final score penalizes missing prediction coverage.
sodium_case_df["predicted_transfer_score_filled"] = sodium_case_df["predicted_transfer_score"].fillna(sodium_case_df["computed_target_score"])

# Uncertainty penalty from normalized tree std and conformal width.
unc_parts = []
for target in TRIAGE_TARGETS:
    # Use score range as approximate scale.
    cfg = SCORE_RANGES[target]
    if cfg["direction"] == "benefit":
        scale = cfg["high"] - cfg["low"]
    else:
        scale = cfg["bad"] - cfg["good"]
    scale = max(float(scale), 1e-12)

    std_col = f"unc_tree_std_{target}"
    width_col = f"conf_width_{target}"

    target_unc = []
    if std_col in sodium_case_df.columns:
        target_unc.append((pd.to_numeric(sodium_case_df[std_col], errors="coerce") / scale).clip(0, 2))
    if width_col in sodium_case_df.columns:
        # A 95% width is roughly 3.92 sigma under a Gaussian approximation.
        target_unc.append((pd.to_numeric(sodium_case_df[width_col], errors="coerce") / (2.0 * scale)).clip(0, 2))

    if target_unc:
        unc_parts.append(pd.concat(target_unc, axis=1).mean(axis=1))

if unc_parts:
    sodium_case_df["uncertainty_penalty_raw"] = pd.concat(unc_parts, axis=1).mean(axis=1)
else:
    sodium_case_df["uncertainty_penalty_raw"] = np.nan

# Normalize uncertainty penalty within Na set.
unc = sodium_case_df["uncertainty_penalty_raw"]
if unc.notna().sum() > 1 and unc.max() > unc.min():
    sodium_case_df["uncertainty_penalty_norm"] = ((unc - unc.min()) / (unc.max() - unc.min())).clip(0, 1)
else:
    sodium_case_df["uncertainty_penalty_norm"] = 0.5

# Applicability-domain penalty from AD flags and domain percentiles.
ad_parts = []
for target in TRIAGE_TARGETS:
    ad_num_col = f"ad_flag_num_{target}"
    pct_col = f"domain_pct_{target}"
    if ad_num_col in sodium_case_df.columns:
        ad_parts.append((pd.to_numeric(sodium_case_df[ad_num_col], errors="coerce") / 2.0).clip(0, 1))
    elif pct_col in sodium_case_df.columns:
        ad_parts.append((pd.to_numeric(sodium_case_df[pct_col], errors="coerce") / 100.0).clip(0, 1))

if ad_parts:
    sodium_case_df["ad_penalty_norm"] = pd.concat(ad_parts, axis=1).mean(axis=1).fillna(0.5)
else:
    sodium_case_df["ad_penalty_norm"] = 0.5

# Missing prediction penalty.
sodium_case_df["missing_prediction_penalty"] = 1.0 - (
    sodium_case_df["prediction_coverage_n_targets"] / max(len(TRIAGE_TARGETS), 1)
).clip(0, 1)

# Transparent uncertainty/application-domain penalized triage score.
sodium_case_df["uncertainty_penalized_score"] = (
    0.45 * sodium_case_df["computed_target_score"].fillna(0)
    + 0.45 * sodium_case_df["predicted_transfer_score_filled"].fillna(0)
    - 0.06 * sodium_case_df["uncertainty_penalty_norm"].fillna(0.5)
    - 0.04 * sodium_case_df["ad_penalty_norm"].fillna(0.5)
    - 0.05 * sodium_case_df["missing_prediction_penalty"].fillna(1.0)
)

# Do not reward records that fail the case-study screen; keep them visible but demote them.
sodium_case_df["screened_uncertainty_penalized_score"] = sodium_case_df["uncertainty_penalized_score"]
sodium_case_df.loc[~sodium_case_df["case_screen_pass"].fillna(False), "screened_uncertainty_penalized_score"] -= 0.25

# Rank columns.
sodium_case_df["rank_computed_score"] = sodium_case_df["computed_target_score"].rank(ascending=False, method="min")
sodium_case_df["rank_predicted_transfer_score"] = sodium_case_df["predicted_transfer_score_filled"].rank(ascending=False, method="min")
sodium_case_df["rank_uncertainty_penalized_score"] = sodium_case_df["screened_uncertainty_penalized_score"].rank(ascending=False, method="min")

print("Computed and uncertainty-penalized triage scores added.")
display(sodium_case_df[[
    "battery_formula", "framework_formula", "coarse_family", "average_voltage", "capacity_grav", "energy_grav",
    "max_delta_volume", "stability_worst", "case_screen_pass", "computed_target_score",
    "predicted_transfer_score", "uncertainty_penalty_norm", "ad_penalty_norm", "screened_uncertainty_penalized_score"
]].sort_values("screened_uncertainty_penalized_score", ascending=False).head(10))


In [ ]:
# ============================================================
# Cell 8 — Pareto membership and non-dominated fronts
# ============================================================

def pareto_fronts(values: np.ndarray) -> np.ndarray:
    """
    Non-dominated sorting. Assumes all objectives are to maximize.
    Returns front number: 1 is non-dominated Pareto front.
    O(n^3) worst-case but fine for Na subset size.
    """
    n = values.shape[0]
    remaining = set(range(n))
    fronts = np.full(n, np.nan)
    front_id = 1

    # Replace NaN with very poor values.
    arr = np.array(values, dtype=float)
    arr[~np.isfinite(arr)] = -np.inf

    while remaining:
        rem = list(remaining)
        current_front = []

        for i in rem:
            dominated = False
            for j in rem:
                if i == j:
                    continue
                # j dominates i if j >= i on all objectives and > i on at least one.
                if np.all(arr[j] >= arr[i]) and np.any(arr[j] > arr[i]):
                    dominated = True
                    break
            if not dominated:
                current_front.append(i)

        for i in current_front:
            fronts[i] = front_id
            remaining.remove(i)

        front_id += 1
        if front_id > n + 1:
            break

    return fronts.astype(int)

# Pareto on computed MP targets.
pareto_cols_computed = [
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "neg_max_delta_volume",
    "neg_stability_worst",
]
pareto_input = sodium_case_df.copy()
pareto_input["neg_max_delta_volume"] = -pd.to_numeric(pareto_input["max_delta_volume"], errors="coerce")
pareto_input["neg_stability_worst"] = -pd.to_numeric(pareto_input["stability_worst"], errors="coerce")

computed_values = pareto_input[pareto_cols_computed].to_numpy(dtype=float)
sodium_case_df["computed_pareto_front"] = pareto_fronts(computed_values)
sodium_case_df["computed_pareto_member"] = sodium_case_df["computed_pareto_front"] == 1

# Pareto on predicted transfer targets where available.
pred_pareto_df = pd.DataFrame(index=sodium_case_df.index)
for target in TRIAGE_TARGETS:
    pred_col = f"pred_{target}"
    if pred_col in sodium_case_df.columns:
        pred_pareto_df[target] = pd.to_numeric(sodium_case_df[pred_col], errors="coerce")
    else:
        pred_pareto_df[target] = np.nan
pred_pareto_df["neg_max_delta_volume"] = -pred_pareto_df["max_delta_volume"]
pred_pareto_df["neg_stability_worst"] = -pred_pareto_df["stability_worst"]
predicted_values = pred_pareto_df[["average_voltage", "capacity_grav", "energy_grav", "neg_max_delta_volume", "neg_stability_worst"]].to_numpy(dtype=float)

# If a row has no prediction coverage, it will fall into a poor front automatically.
sodium_case_df["predicted_pareto_front"] = pareto_fronts(predicted_values)
sodium_case_df["predicted_pareto_member"] = sodium_case_df["predicted_pareto_front"] == 1

# Add a simple Pareto support score.
sodium_case_df["pareto_support_score"] = (
    sodium_case_df["computed_pareto_member"].astype(int) * 0.6
    + sodium_case_df["predicted_pareto_member"].astype(int) * 0.4
)

# Add Pareto support into final triage score with small weight.
sodium_case_df["final_triage_score"] = (
    sodium_case_df["screened_uncertainty_penalized_score"]
    + 0.05 * sodium_case_df["pareto_support_score"]
)

sodium_case_df["final_triage_rank"] = sodium_case_df["final_triage_score"].rank(ascending=False, method="min")

pareto_df = sodium_case_df[
    sodium_case_df["computed_pareto_member"] | sodium_case_df["predicted_pareto_member"]
].copy().sort_values("final_triage_score", ascending=False)

pareto_path = PROCESSED_DIR / "08_sodium_pareto_candidates.csv"
pareto_df.to_csv(pareto_path, index=False)

pareto_summary_df = pd.DataFrame([{
    "n_sodium_records": len(sodium_case_df),
    "n_computed_pareto_front": int(sodium_case_df["computed_pareto_member"].sum()),
    "n_predicted_pareto_front": int(sodium_case_df["predicted_pareto_member"].sum()),
    "n_any_pareto_support": int((sodium_case_df["computed_pareto_member"] | sodium_case_df["predicted_pareto_member"]).sum()),
}])
pareto_summary_df.to_csv(AUDIT_DIR / "08_pareto_summary.csv", index=False)

display(pareto_summary_df)
print(f"Saved Pareto candidates: {pareto_path}")


In [ ]:
# ============================================================
# Cell 9 — Monte-Carlo rank robustness using conformal intervals
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# Candidate pool for rank robustness: all sodium records, but failed screen records are penalized.
mc_df = sodium_case_df.copy().reset_index(drop=True)
n = len(mc_df)

# Precompute computed score and penalties.
computed_score = mc_df["computed_target_score"].fillna(0.0).to_numpy(float)
unc_penalty = mc_df["uncertainty_penalty_norm"].fillna(0.5).to_numpy(float)
ad_penalty = mc_df["ad_penalty_norm"].fillna(0.5).to_numpy(float)
missing_pred_penalty = mc_df["missing_prediction_penalty"].fillna(1.0).to_numpy(float)
case_screen_penalty = np.where(mc_df["case_screen_pass"].fillna(False).to_numpy(bool), 0.0, 0.25)
pareto_support = mc_df["pareto_support_score"].fillna(0.0).to_numpy(float)

rank_records = np.zeros((RANK_MC_SAMPLES, n), dtype=np.int32)
score_records_sample_mean = np.zeros(n, dtype=float)

# Helpers for MC scoring.
def normalize_array_for_target(arr: np.ndarray, target: str) -> np.ndarray:
    cfg = SCORE_RANGES[target]
    arr = np.array(arr, dtype=float)
    if cfg["direction"] == "benefit":
        out = (arr - cfg["low"]) / (cfg["high"] - cfg["low"])
    else:
        out = 1.0 - ((arr - cfg["good"]) / (cfg["bad"] - cfg["good"]))
    return np.clip(out, 0.0, 1.0)

# Build prediction mu/sigma arrays. Missing prediction falls back to computed value with large missing penalty.
pred_mu = {}
pred_sigma = {}
for target in TRIAGE_TARGETS:
    pred_col = f"pred_{target}"
    conf_width_col = f"conf_width_{target}"
    tree_std_col = f"unc_tree_std_{target}"

    mu = pd.to_numeric(mc_df[pred_col], errors="coerce") if pred_col in mc_df.columns else pd.Series(np.nan, index=mc_df.index)
    fallback = pd.to_numeric(mc_df[target], errors="coerce") if target in mc_df.columns else pd.Series(np.nan, index=mc_df.index)
    mu = mu.fillna(fallback).fillna(0.0).to_numpy(float)

    if conf_width_col in mc_df.columns:
        width = pd.to_numeric(mc_df[conf_width_col], errors="coerce")
        sigma = (width / 3.92).replace([np.inf, -np.inf], np.nan)
    else:
        sigma = pd.Series(np.nan, index=mc_df.index)

    if tree_std_col in mc_df.columns:
        tree_std = pd.to_numeric(mc_df[tree_std_col], errors="coerce")
        sigma = sigma.fillna(tree_std)

    # Conservative fallback scale if no uncertainty exists.
    cfg = SCORE_RANGES[target]
    if cfg["direction"] == "benefit":
        fallback_sigma = 0.10 * (cfg["high"] - cfg["low"])
    else:
        fallback_sigma = 0.10 * (cfg["bad"] - cfg["good"])

    sigma = sigma.fillna(fallback_sigma).clip(lower=1e-12).to_numpy(float)
    pred_mu[target] = mu
    pred_sigma[target] = sigma

for s in range(RANK_MC_SAMPLES):
    sampled_components = []
    sampled_weights = []

    for target, weight in SCORE_WEIGHTS.items():
        mu = pred_mu[target]
        sigma = pred_sigma[target]
        sampled = rng.normal(mu, sigma)

        # Physical clipping for stability of MC.
        if target in ["capacity_grav", "energy_grav", "max_delta_volume", "stability_worst"]:
            sampled = np.clip(sampled, 0.0, None)
        if target == "average_voltage":
            sampled = np.clip(sampled, 0.0, 6.0)

        comp = normalize_array_for_target(sampled, target)
        sampled_components.append(weight * comp)
        sampled_weights.append(weight)

    pred_sample_score = np.sum(sampled_components, axis=0) / max(sum(sampled_weights), 1e-12)

    sampled_final_score = (
        0.45 * computed_score
        + 0.45 * pred_sample_score
        - 0.06 * unc_penalty
        - 0.04 * ad_penalty
        - 0.05 * missing_pred_penalty
        - case_screen_penalty
        + 0.05 * pareto_support
    )

    score_records_sample_mean += sampled_final_score

    # Rank 1 = highest score.
    order = np.argsort(-sampled_final_score)
    ranks = np.empty(n, dtype=np.int32)
    ranks[order] = np.arange(1, n + 1)
    rank_records[s, :] = ranks

score_records_sample_mean /= RANK_MC_SAMPLES

rank_mean = rank_records.mean(axis=0)
rank_median = np.median(rank_records, axis=0)
rank_std = rank_records.std(axis=0)
rank_q25 = np.percentile(rank_records, 25, axis=0)
rank_q75 = np.percentile(rank_records, 75, axis=0)
rank_iqr = rank_q75 - rank_q25

top10_probability = (rank_records <= 10).mean(axis=0)
top20_probability = (rank_records <= 20).mean(axis=0)
top30_probability = (rank_records <= 30).mean(axis=0)

mc_df["mc_score_mean"] = score_records_sample_mean
mc_df["rank_mean_mc"] = rank_mean
mc_df["rank_median_mc"] = rank_median
mc_df["rank_std_mc"] = rank_std
mc_df["rank_iqr_mc"] = rank_iqr
mc_df["top10_probability_mc"] = top10_probability
mc_df["top20_probability_mc"] = top20_probability
mc_df["top30_probability_mc"] = top30_probability

# Merge rank robustness columns back.
rank_cols = [
    "electrode_uid",
    "mc_score_mean",
    "rank_mean_mc",
    "rank_median_mc",
    "rank_std_mc",
    "rank_iqr_mc",
    "top10_probability_mc",
    "top20_probability_mc",
    "top30_probability_mc",
]
sodium_case_df = sodium_case_df.merge(mc_df[rank_cols], on="electrode_uid", how="left")

rank_robustness_df = sodium_case_df.sort_values(
    ["top20_probability_mc", "top10_probability_mc", "mc_score_mean"],
    ascending=[False, False, False],
).copy()

rank_robustness_path = PROCESSED_DIR / "08_sodium_rank_robustness.csv"
rank_robustness_df.to_csv(rank_robustness_path, index=False)

rank_audit_df = pd.DataFrame([{
    "rank_mc_samples": RANK_MC_SAMPLES,
    "n_sodium_records": n,
    "n_top10_probability_gt_0": int((top10_probability > 0).sum()),
    "n_top20_probability_gt_0": int((top20_probability > 0).sum()),
    "n_top30_probability_gt_0": int((top30_probability > 0).sum()),
    "mean_rank_iqr": float(np.mean(rank_iqr)),
    "median_rank_iqr": float(np.median(rank_iqr)),
}])
rank_audit_df.to_csv(AUDIT_DIR / "08_rank_robustness_audit.csv", index=False)

display(rank_audit_df)
print(f"Saved rank robustness table: {rank_robustness_path}")


In [ ]:
# ============================================================
# Cell 10 — Build sodium case-study final candidate table and shortlists
# ============================================================

# Final sort: robust top-20 probability first, then final triage score.
sodium_case_df["shortlist_priority_score"] = (
    0.50 * sodium_case_df["top20_probability_mc"].fillna(0.0)
    + 0.25 * sodium_case_df["top10_probability_mc"].fillna(0.0)
    + 0.25 * sodium_case_df["mc_score_mean"].fillna(sodium_case_df["final_triage_score"].fillna(0.0))
)

# Additional audit flags.
sodium_case_df["high_uncertainty_flag"] = sodium_case_df["uncertainty_penalty_norm"] >= 0.75
sodium_case_df["out_of_domain_flag"] = sodium_case_df["ad_penalty_norm"] >= 0.75
sodium_case_df["case_study_do_not_overclaim_flag"] = (
    sodium_case_df["high_uncertainty_flag"]
    | sodium_case_df["out_of_domain_flag"]
    | (~sodium_case_df["case_screen_pass"].fillna(False))
)

# Keep all records in the main candidate table.
main_sort_cols = ["shortlist_priority_score", "final_triage_score", "computed_target_score"]
sodium_case_df = sodium_case_df.sort_values(main_sort_cols, ascending=[False, False, False]).copy()
sodium_case_df["shortlist_priority_rank"] = np.arange(1, len(sodium_case_df) + 1)

# Recommended manual review shortlist.
manual_review_df = sodium_case_df[
    sodium_case_df["case_screen_pass"].fillna(False)
    & sodium_case_df["prediction_coverage_complete_primary"].fillna(False)
].copy()

manual_review_df = manual_review_df.sort_values(
    ["top20_probability_mc", "top10_probability_mc", "final_triage_score", "computed_target_score"],
    ascending=[False, False, False, False],
)

# Top 20/30 for manual literature-analogue validation in Notebook 09.
top20_manual_review_df = manual_review_df.head(20).copy()
top30_manual_review_df = manual_review_df.head(30).copy()

# Uncertainty-penalized shortlist table.
uncertainty_penalized_shortlist_df = sodium_case_df[
    sodium_case_df["case_screen_pass"].fillna(False)
].sort_values("final_triage_score", ascending=False).head(50).copy()

# Select display/export columns where available.
preferred_cols = [
    "shortlist_priority_rank",
    "record_index",
    "electrode_uid",
    "battery_formula",
    "formula_charge",
    "formula_discharge",
    "framework_formula",
    "coarse_family",
    "chemical_system_uid",
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_worst",
    "case_screen_pass",
    "computed_pareto_member",
    "predicted_pareto_member",
    "computed_target_score",
    "predicted_transfer_score",
    "uncertainty_penalty_norm",
    "ad_penalty_norm",
    "final_triage_score",
    "mc_score_mean",
    "rank_median_mc",
    "rank_iqr_mc",
    "top10_probability_mc",
    "top20_probability_mc",
    "top30_probability_mc",
    "prediction_coverage_n_targets",
    "high_uncertainty_flag",
    "out_of_domain_flag",
    "case_study_do_not_overclaim_flag",
]

# Include prediction columns.
for target in TRIAGE_TARGETS:
    for prefix in ["pred_", "unc_tree_std_", "conf_lower_", "conf_upper_", "conf_width_", "ad_flag_"]:
        col = f"{prefix}{target}"
        if col in sodium_case_df.columns:
            preferred_cols.append(col)

preferred_cols = [c for c in preferred_cols if c in sodium_case_df.columns]

candidate_table_path = PROCESSED_DIR / "08_sodium_case_study_candidate_table.csv"
sodium_case_df.to_csv(candidate_table_path, index=False)

candidate_table_compact_path = PROCESSED_DIR / "08_sodium_case_study_candidate_table_compact.csv"
sodium_case_df[preferred_cols].to_csv(candidate_table_compact_path, index=False)

uncertainty_shortlist_path = PROCESSED_DIR / "08_sodium_uncertainty_penalized_shortlist.csv"
uncertainty_penalized_shortlist_df[preferred_cols].to_csv(uncertainty_shortlist_path, index=False)

top20_path = PROCESSED_DIR / "08_sodium_top20_candidates_for_literature_review.csv"
top20_manual_review_df[preferred_cols].to_csv(top20_path, index=False)

top30_path = PROCESSED_DIR / "08_sodium_top30_candidates_for_literature_review.csv"
top30_manual_review_df[preferred_cols].to_csv(top30_path, index=False)

shortlist_summary_df = pd.DataFrame([{
    "n_sodium_records": len(sodium_case_df),
    "n_case_screen_pass": int(sodium_case_df["case_screen_pass"].sum()),
    "n_prediction_complete_primary": int(sodium_case_df["prediction_coverage_complete_primary"].sum()),
    "n_manual_review_pool": len(manual_review_df),
    "n_top20_manual_review": len(top20_manual_review_df),
    "n_top30_manual_review": len(top30_manual_review_df),
    "n_uncertainty_penalized_shortlist_50": len(uncertainty_penalized_shortlist_df),
}])
shortlist_summary_df.to_csv(AUDIT_DIR / "08_shortlist_summary.csv", index=False)

display(shortlist_summary_df)
display(top20_manual_review_df[preferred_cols].head(20))

print(f"Saved full candidate table: {candidate_table_path}")
print(f"Saved compact candidate table: {candidate_table_compact_path}")
print(f"Saved uncertainty-penalized shortlist: {uncertainty_shortlist_path}")
print(f"Saved top-20 manual-review candidates: {top20_path}")
print(f"Saved top-30 manual-review candidates: {top30_path}")


In [ ]:
# ============================================================
# Cell 11 — Family and chemistry summaries for Notebook 09 planning
# ============================================================

family_summary_df = (
    sodium_case_df
    .groupby("coarse_family", dropna=False)
    .agg(
        n_records=("electrode_uid", "count"),
        n_case_screen_pass=("case_screen_pass", "sum"),
        n_computed_pareto=("computed_pareto_member", "sum"),
        n_predicted_pareto=("predicted_pareto_member", "sum"),
        median_final_triage_score=("final_triage_score", "median"),
        max_final_triage_score=("final_triage_score", "max"),
        median_uncertainty_penalty=("uncertainty_penalty_norm", "median"),
        median_ad_penalty=("ad_penalty_norm", "median"),
    )
    .reset_index()
    .sort_values("max_final_triage_score", ascending=False)
)
family_summary_df.to_csv(AUDIT_DIR / "08_family_level_triage_summary.csv", index=False)

chemsys_summary_df = (
    sodium_case_df
    .groupby("chemical_system_uid", dropna=False)
    .agg(
        n_records=("electrode_uid", "count"),
        n_case_screen_pass=("case_screen_pass", "sum"),
        best_final_triage_score=("final_triage_score", "max"),
        best_top20_probability=("top20_probability_mc", "max"),
        families=("coarse_family", lambda x: "|".join(sorted(set(map(str, x))))),
    )
    .reset_index()
    .sort_values("best_final_triage_score", ascending=False)
)
chemsys_summary_df.to_csv(AUDIT_DIR / "08_chemical_system_triage_summary.csv", index=False)

# Top formula list for Notebook 09 search planning.
formula_review_df = top30_manual_review_df[[
    c for c in [
        "shortlist_priority_rank",
        "battery_formula",
        "formula_discharge",
        "framework_formula",
        "coarse_family",
        "chemical_system_uid",
        "average_voltage",
        "capacity_grav",
        "energy_grav",
        "max_delta_volume",
        "stability_worst",
        "top20_probability_mc",
        "final_triage_score",
        "case_study_do_not_overclaim_flag",
    ] if c in top30_manual_review_df.columns
]].copy()
formula_review_df.to_csv(PROCESSED_DIR / "08_formula_list_for_notebook_09_manual_search.csv", index=False)

display(family_summary_df)
display(formula_review_df.head(30))


In [ ]:
# ============================================================
# Cell 12 — Reviewer-safety audit for the sodium case study
# ============================================================

reviewer_safety_rows = []

# Confirm no CDE/criticality columns were introduced.
forbidden_patterns = [
    "cde",
    "chemdataextractor",
    "criticality",
    "critical",
    "usgs",
    "crma",
    "bgs",
    "iea",
    "topsis",
]

all_cols_lower = [c.lower() for c in sodium_case_df.columns]
for pattern in forbidden_patterns:
    hits = [c for c in sodium_case_df.columns if pattern in c.lower()]
    reviewer_safety_rows.append({
        "check": f"forbidden_pattern_{pattern}",
        "status": "PASS" if len(hits) == 0 else "FAIL",
        "details": "none" if len(hits) == 0 else "|".join(hits[:20]),
    })

# Enough manual review candidates.
n_manual_pool = len(manual_review_df)
reviewer_safety_rows.append({
    "check": "manual_review_pool_at_least_20",
    "status": "PASS" if n_manual_pool >= 20 else "FAIL",
    "details": f"n_manual_review_pool={n_manual_pool}",
})

# Prediction coverage.
pred_complete_pct = 100.0 * sodium_case_df["prediction_coverage_complete_primary"].sum() / max(len(sodium_case_df), 1)
reviewer_safety_rows.append({
    "check": "prediction_coverage_complete_primary_at_least_70pct",
    "status": "PASS" if pred_complete_pct >= 70.0 else "CONDITIONAL_PASS",
    "details": f"complete_primary_prediction_pct={pred_complete_pct:.2f}",
})

# Overclaim flags in top 20.
if len(top20_manual_review_df) > 0:
    top20_overclaim_pct = 100.0 * top20_manual_review_df["case_study_do_not_overclaim_flag"].sum() / len(top20_manual_review_df)
else:
    top20_overclaim_pct = 100.0
reviewer_safety_rows.append({
    "check": "top20_overclaim_flag_rate_reported",
    "status": "PASS",
    "details": f"top20_do_not_overclaim_flag_pct={top20_overclaim_pct:.2f}",
})

# Case screen count.
n_case_pass = int(sodium_case_df["case_screen_pass"].sum())
reviewer_safety_rows.append({
    "check": "case_screen_pass_at_least_30",
    "status": "PASS" if n_case_pass >= 30 else "CONDITIONAL_PASS",
    "details": f"n_case_screen_pass={n_case_pass}",
})

reviewer_safety_df = pd.DataFrame(reviewer_safety_rows)
reviewer_safety_df.to_csv(AUDIT_DIR / "08_reviewer_safety_audit.csv", index=False)

display(reviewer_safety_df)

n_safety_fail = int((reviewer_safety_df["status"] == "FAIL").sum())
n_safety_conditional = int((reviewer_safety_df["status"] == "CONDITIONAL_PASS").sum())

if n_safety_fail > 0:
    log_event("reviewer_safety", "ERROR", "Reviewer-safety audit has FAIL rows.", {"n_fail": n_safety_fail})
elif n_safety_conditional > 0:
    log_event("reviewer_safety", "WARNING", "Reviewer-safety audit has conditional rows.", {"n_conditional": n_safety_conditional})
else:
    log_event("reviewer_safety", "INFO", "Reviewer-safety audit passed.")

save_event_log()


In [ ]:
# ============================================================
# Cell 13 — Final decision and output manifest
# ============================================================

def list_output_files(base_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(base_dir)),
                "size_bytes": path.stat().st_size,
                "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    return pd.DataFrame(rows)

required_outputs = [
    PROCESSED_DIR / "08_sodium_candidate_base_table.csv",
    PROCESSED_DIR / "08_sodium_case_study_candidate_table.csv",
    PROCESSED_DIR / "08_sodium_case_study_candidate_table_compact.csv",
    PROCESSED_DIR / "08_sodium_pareto_candidates.csv",
    PROCESSED_DIR / "08_sodium_rank_robustness.csv",
    PROCESSED_DIR / "08_sodium_uncertainty_penalized_shortlist.csv",
    PROCESSED_DIR / "08_sodium_top20_candidates_for_literature_review.csv",
    PROCESSED_DIR / "08_sodium_top30_candidates_for_literature_review.csv",
    PROCESSED_DIR / "08_formula_list_for_notebook_09_manual_search.csv",
    AUDIT_DIR / "08_input_coverage_audit.csv",
    AUDIT_DIR / "08_sodium_screening_funnel.csv",
    AUDIT_DIR / "08_prediction_coverage_by_target.csv",
    AUDIT_DIR / "08_rank_robustness_audit.csv",
    AUDIT_DIR / "08_reviewer_safety_audit.csv",
    AUDIT_DIR / "08_no_cde_criticality_assertion.json",
]

missing_outputs = [str(p) for p in required_outputs if not p.exists()]

n_sodium = len(sodium_case_df)
n_top20 = len(top20_manual_review_df)
n_case_pass = int(sodium_case_df["case_screen_pass"].sum()) if "case_screen_pass" in sodium_case_df.columns else 0
n_safety_fail = int((reviewer_safety_df["status"] == "FAIL").sum())
n_safety_conditional = int((reviewer_safety_df["status"] == "CONDITIONAL_PASS").sum())

if missing_outputs:
    FINAL_DECISION_12 = "NO_GO_FIX_NOTEBOOK_08_OUTPUTS"
elif n_safety_fail > 0:
    FINAL_DECISION_12 = "NO_GO_FIX_CASE_STUDY_TRIAGE"
elif n_top20 < 20 or n_case_pass < 30 or n_safety_conditional > 0:
    FINAL_DECISION_12 = "CONDITIONAL_GO_TO_NOTEBOOK_09_REVIEW_CAVEATS"
else:
    FINAL_DECISION_12 = "FULL_GO_TO_NOTEBOOK_13"

final_decision_12 = {
    "final_decision": FINAL_DECISION_12,
    "n_sodium_records": n_sodium,
    "n_case_screen_pass": n_case_pass,
    "n_top20_manual_review": n_top20,
    "n_missing_outputs": len(missing_outputs),
    "missing_outputs": missing_outputs,
    "n_reviewer_safety_fail": n_safety_fail,
    "n_reviewer_safety_conditional": n_safety_conditional,
    "ml_training_performed": False,
    "cde_matching_performed": False,
    "criticality_filtering_performed": False,
    "manuscript_writing_performed": False,
    "dft_candidate_selection_performed": False,
    "discovery_claim_made": False,
}

write_json_safe(final_decision_12, METADATA_DIR / "08_final_decision.json")

software_environment = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "notebook_name": "_CMT_PUBLIC_NOTEBOOK_FILENAME_01_",
}
write_json_safe(software_environment, METADATA_DIR / "08_software_environment.json")

output_manifest_df = list_output_files(BASE_DIR)
output_manifest_df.to_csv(METADATA_DIR / "08_output_file_manifest.csv", index=False)

save_event_log()

display(output_manifest_df)

print("\n" + "=" * 80)
print(f"Notebook 08 FINAL DECISION: {FINAL_DECISION_12}")
print("=" * 80)
print("\nKey outputs:")
for p in required_outputs:
    print(f" - {p}  {'[OK]' if p.exists() else '[MISSING]'}")

print("\nNotebook 12 complete.")
